In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from blimpy import Waterfall

# -----------------------
# CONFIG
# -----------------------

ROOT = Path('/home/wlll2x/turboSETI')
CSV_PATH = ROOT / 'scripts' / 'carmenes_1688mhz_subset.csv'

TARGETS = ["NGC5638", "SAG_DIR"]

# -----------------------
# LOAD CSV
# -----------------------

df = pd.read_csv(CSV_PATH)
df["Target"] = df["Target"].astype(str).str.strip()

print("\n=== TARGET CHECK ===")
for t in TARGETS:
    print(f"{t}: {(df['Target'] == t).sum()} rows")

df = df[df["Target"].isin(TARGETS)].copy()

df["Frequency"] = pd.to_numeric(df["Frequency"], errors="coerce")

print("\nFiltered rows:", len(df))

# -----------------------
# FILE RESOLVER
# -----------------------

def resolve_path(row):
    p = Path(str(row[".h5 path"]).strip())

    if p.exists():
        return p

    alt = Path("/datax/scratch/wlll2x/raw") / p.name
    if alt.exists():
        return alt

    return None

# -----------------------
# PICK SAMPLE ROW
# -----------------------

row = df[df[".h5 path"].notna()].iloc[0]

path = resolve_path(row)

if path is None:
    raise FileNotFoundError("Could not resolve HDF5 file")

print("\n=== SELECTED FILE ===")
print("Target:", row["Target"])
print("Frequency (MHz):", row["Frequency"])
print("File:", path)

# -----------------------
# OPEN WATERFALL
# -----------------------

wf = Waterfall(str(path), load_data=True)

fch1 = wf.header.get("fch1")
foff = wf.header.get("foff")
nchans = wf.header.get("nchans")
tsamp = wf.header.get("tsamp")

print("\n=== HEADER ===")
print("fch1:", fch1)
print("foff:", foff)
print("nchans:", nchans)
print("tsamp:", tsamp)

# -----------------------
# BANDWIDTH CHECK
# -----------------------

bandwidth_mhz = abs(foff) * nchans
print("\n=== BANDWIDTH ===")
print("Estimated bandwidth (MHz):", bandwidth_mhz)

# -----------------------
# BASIC DATA CHECK
# -----------------------

data = wf.data

if data.ndim == 3:
    data = data[:, 0, :]

print("\n=== DATA ===")
print("shape:", data.shape)
print("min/max:", np.min(data), np.max(data))

# -----------------------
# QUICK PLOT (SAFE SCALE)
# -----------------------

def quick_plot(data, title="waterfall"):
    plt.figure(figsize=(10, 4))
    plt.imshow(
        10 * np.log10(np.clip(data, 1e-6, None)),
        aspect='auto',
        origin='lower',
        vmin=None,
        vmax=None
    )
    plt.title(title)
    plt.xlabel("Frequency bins")
    plt.ylabel("Time")
    plt.colorbar(label="dB")
    plt.tight_layout()
    plt.show()

quick_plot(data, f"{row['Target']} quick look")

print("\nDONE")


=== TARGET CHECK ===
NGC5638: 0 rows
SAG_DIR: 6 rows

Filtered rows: 6

=== SELECTED FILE ===
Target: SAG_DIR
Frequency (MHz): 1688
File: /datag/pipeline/AGBT21A_996_46/blc03/blc03_guppi_59404_23767_Sag_dIr_0094.rawspec.0000.h5

=== HEADER ===
fch1: 1688.96484375
foff: -2.7939677238464355e-06
nchans: 67108864
tsamp: 18.253611007999982

=== BANDWIDTH ===
Estimated bandwidth (MHz): 187.5

=== DATA ===
shape: (16, 67108864)
min/max: 2218687.5 570452740000.0

DONE
